To extract the unique categories from the OSM and Foursquare datasets, you can use the following SQL commands. These commands create new tables containing distinct category information and export them to CSV files.

```sql
CREATE TABLE osm_categories AS
SELECT DISTINCT osm_class, osm_type FROM osm ;
\copy osm_categories TO 'osm_categories.csv' WITH ( FORMAT csv, HEADER, DELIMITER ',', QUOTE '"', ESCAPE '"', NULL '' );
```


Foursquare can be downloaded from the Foursquare website: https://docs.foursquare.com/data-products/docs/categories
(data/fsq_personalization-apis-movement-sdk-categories.csv)

Then we feed data/fsq_personalization-apis-movement-sdk-categories.csv, osm_categories.csv and https://wiki.openstreetmap.org/wiki/Map_features to `GPT 5.5 with High Intelligence` to generate a mapping between Foursquare categories and OSM categories. Resulted in a mapping file fsq_osm_category_mapping.csv. 

prompt: "I want you to add two columns to fsq_personalization-apis-movement-sdk-categories.csv that are osm_class, osm_type that they have the same meaning. If ou cannot find a match from osm please put nf_fsq for both osm_class, osm_type."

In [21]:
import pandas
fsq_osm_category_mapping = pandas.read_csv('../../data/fsq_osm_category_mapping.csv')
fsq_personalization = pandas.read_csv('../../data/fsq_personalization-apis-movement-sdk-categories.csv')

In [22]:

# check that the two files have the same [Category ID, Category Name, Category Label] values 
f1 = fsq_osm_category_mapping[['Category ID', 'Category Name', 'Category Label']].sort_values(by=['Category ID', 'Category Name', 'Category Label']).reset_index(drop=True)
f2 = fsq_personalization[['Category ID', 'Category Name', 'Category Label']].sort_values(by=['Category ID', 'Category Name', 'Category Label']).reset_index(drop=True)
assert f1.equals(f2), "The two files do not have the same [Category ID, Category Name, Category Label] values. Please check the files for discrepancies."

In [23]:
category_mapping_dict = {}
for index, row in fsq_osm_category_mapping.iterrows():
    category_mapping_dict[row['Category ID']] = {
                                            'Category Name': row['Category Name'],
                                            'Category Label': row['Category Label'],
                                            "osm_class": row['osm_class'],
                                             "osm_type": row['osm_type']}


In [24]:
# category_mapping_dict:
# {'4d4b7104d754a06370d81259': {'osm_class': 'amenity', 'osm_type': 'yes'},

In [25]:
world_poi_df = pandas.read_csv('../../data/World_POI_levenshtein_0.3.csv', chunksize=100_000)

In [26]:
def is_similar(category_dict, row):
    res=False
    if category_dict['osm_class'] == row['osm_class']:
        res = True
    elif category_dict['osm_type'] == row['osm_type']:
        res = True
                
    return res
                        
counter = 0
total = 0
all_candidate={}
for chunk in world_poi_df:
    # print(chunk.columns)
    chunk.dropna(subset=['fsq_category_ids'], inplace=True)
    for index, row in chunk.iterrows():
        if row['fsq_osm_name_similarity_score_trg'] == 1.0:
            if row['fsq_category_ids'] != '':
                row['fsq_category_ids'] = row['fsq_category_ids'].replace(' ', ', ')
                cat_ids = eval(row['fsq_category_ids'])
                for cat_id in cat_ids:
                    if cat_id in category_mapping_dict.keys():
                        if is_similar(category_mapping_dict[cat_id], row):
                            counter += 1
                        if cat_id not in all_candidate:
                            all_candidate[cat_id] = {"osm_class": set(), "osm_type": set()}
                        all_candidate[cat_id]['osm_type'].add(row['osm_type'])
                        all_candidate[cat_id]['osm_class'].add(row['osm_class'])
                        total += 1
    break                 
print(f"counter: {counter}, total: {total}, percentage: {counter/total*100:.2f}%")

/tmp/ipykernel_380835/1901503942.py:13: DtypeWarning: Columns (0: fsq_post_town) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in world_poi_df:


counter: 20370, total: 26816, percentage: 75.96%


In [29]:
print("Category ID,Category Name,Category Label,osm_class,osm_type")
for cid, v in all_candidate.items():
    print(f"{cid},{category_mapping_dict[cid]['Category Name']},{category_mapping_dict[cid]['Category Label']},{v['osm_class']},{v['osm_type']}")

Category ID,Category Name,Category Label,osm_class,osm_type
4bf58dd8d48988d17c941735,Casino,Arts and Entertainment > Casino,{'historic', 'leisure', 'shop', 'tourism', 'amenity'},{'adult_gaming_centre', 'hotel', 'marina', 'bar', 'kiosk', 'manor'}
4bf58dd8d48988d178941735,Dentist,Health and Medicine > Dentist,{'amenity', 'healthcare', 'building'},{'doctors', 'hospital', 'yes', 'dentist', 'clinic'}
4bf58dd8d48988d15c941735,Cemetery,Community and Government > Cemetery,{'landuse', 'historic', 'information', 'building', 'tourism', 'amenity'},{'guidepost', 'museum', 'hut', 'grave_yard', 'parking', 'cemetery', 'yes', 'map', 'memorial'}
4d4b7105d754a06375d81259,Business and Professional Services,Business and Professional Services,{'landuse', 'place', 'leisure', 'shop', 'healthcare', 'building', 'tourism', 'craft', 'office', 'man_made', 'emergency', 'amenity'},{'public_building', 'bicycle', 'house', 'forestry', 'association', 'social_facility', 'works', 'toolmaker', 'government', 'podiatrist', '